# 02. Get User Ratings

This notebook builds the user-item interaction layer used by the recommendation and graph stages.

Source change: this now uses Hernan4444's Kaggle dataset `hernan4444/anime-recommendation-database-2020`, specifically `rating_complete.csv`.

Why this source:

- the dataset documents its collection process in the linked GitHub notebook
- `rating_complete.csv` contains completed anime only (`watching_status == 2`) with non-zero ratings
- the anime ids are already MAL ids, so no fragile old-id-to-MAL-id mapping is needed

Command-line equivalent:

```bash
python src/02_run_ratings_ingestion.py --execute
```

The processed output keeps the existing project schema: `userID`, `animeID`, `rating`.


In [1]:
import json
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

try:
    import kagglehub
except ImportError as exc:
    raise ImportError(
        "kagglehub is required to download the ratings dataset. "
        "Install it with: pip install kagglehub"
    ) from exc

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
BUILD_DIR = BASE_DIR / "data" / "build"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

for directory in [RAW_DIR, BUILD_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "hernan4444/anime-recommendation-database-2020"
RAW_RATINGS_FILE = RAW_DIR / "kaggle_rating_complete_source.csv"
RATINGS_PROCESSED_FILE = PROCESSED_DIR / "ratings_processed.csv"
ANIME_DATASET_FILE = PROCESSED_DIR / "anime_dataset.csv"
RATINGS_BUILD_SUMMARY_FILE = BUILD_DIR / "ratings_build_summary.json"

# Legacy raw files from the previous ratings source. They are disposable.
LEGACY_RAW_ANIMES_FILE = RAW_DIR / "animes.csv"
LEGACY_RAW_RATINGS_FILE = RAW_DIR / "ratings.csv"
LEGACY_RAW_ANIMES_SOURCE_FILE = RAW_DIR / "kaggle_anime_source.csv"
LEGACY_RAW_RATINGS_SOURCE_FILE = RAW_DIR / "kaggle_user_ratings_source.csv"

CHUNK_SIZE = 2_000_000

print("Working directory:", BASE_DIR)
print("Kaggle dataset:", KAGGLE_DATASET)
print("Processed output:", RATINGS_PROCESSED_FILE)


Working directory: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect
Kaggle dataset: hernan4444/anime-recommendation-database-2020
Processed output: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\ratings_processed.csv


c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download `rating_complete.csv`

The raw Kaggle CSV is treated as disposable. If it is missing locally, the notebook downloads the Kaggle dataset and copies only `rating_complete.csv` into `data/raw/`.

The Kaggle cache may still keep the downloaded zip under the user cache directory, but this project folder only keeps the processed interaction file after the build completes.


In [2]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def atomic_write_json(path, payload):
    path = Path(path)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    tmp_path.replace(path)


def csv_has_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns.str.strip()
    except Exception:
        return False
    return set(required_columns).issubset(set(columns))


def find_rating_complete_file(dataset_dir):
    dataset_dir = Path(dataset_dir)
    preferred = list(dataset_dir.rglob("rating_complete.csv"))
    for candidate in preferred:
        if csv_has_columns(candidate, {"user_id", "anime_id", "rating"}):
            return candidate

    for candidate in dataset_dir.rglob("*.csv"):
        if csv_has_columns(candidate, {"user_id", "anime_id", "rating"}):
            return candidate

    return None


def remove_disposable_raw_files():
    for disposable_file in [
        RAW_RATINGS_FILE,
        LEGACY_RAW_ANIMES_FILE,
        LEGACY_RAW_RATINGS_FILE,
        LEGACY_RAW_ANIMES_SOURCE_FILE,
        LEGACY_RAW_RATINGS_SOURCE_FILE,
    ]:
        if disposable_file.exists():
            disposable_file.unlink()
            print("Deleted disposable raw file:", disposable_file)


def ensure_rating_complete_source():
    if RAW_RATINGS_FILE.exists() and csv_has_columns(RAW_RATINGS_FILE, {"user_id", "anime_id", "rating"}):
        print("Using existing raw rating_complete source:", RAW_RATINGS_FILE)
        return RAW_RATINGS_FILE

    print("Downloading KaggleHub dataset:", KAGGLE_DATASET)
    dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Downloaded dataset path:", dataset_path)

    rating_file = find_rating_complete_file(dataset_path)
    if rating_file is None:
        raise FileNotFoundError(
            "Could not find rating_complete.csv with columns user_id, anime_id, rating "
            f"inside {dataset_path}"
        )

    if rating_file.resolve() != RAW_RATINGS_FILE.resolve():
        shutil.copy2(rating_file, RAW_RATINGS_FILE)
    print("Restored rating_complete source:", RAW_RATINGS_FILE)
    return RAW_RATINGS_FILE


raw_ratings_path = ensure_rating_complete_source()
print(pd.read_csv(raw_ratings_path, nrows=5).head())


100%|██████████| 661M/661M [00:52<00:00, 13.2MB/s] 

Extracting files...


Downloaded dataset path: C:\Users\CHAMPUX\.cache\kagglehub\datasets\hernan4444\anime-recommendation-database-2020\versions\7
Restored rating_complete source: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\kaggle_rating_complete_source.csv
   user_id  anime_id  rating
0        0       430       9
1        0      1004       5
2        0      3010       7
3        0       570       7
4        0      2762       9


## Build Project Ratings File

`rating_complete.csv` already uses MAL anime ids, so the processing step is much simpler than the old ratings source.

This notebook streams the file in chunks so the 57M-row source does not need to fit in RAM. If `anime_dataset.csv` exists, ratings are filtered to anime ids that exist in the current project catalog. That keeps the interaction layer aligned with the recommendation catalog.


In [3]:
catalog_ids = None
if ANIME_DATASET_FILE.exists():
    catalog_ids = set(pd.read_csv(ANIME_DATASET_FILE, usecols=["mal_id"])["mal_id"].dropna().astype("int32"))
    print(f"Catalog filter enabled: {len(catalog_ids):,} anime ids")
else:
    print("Catalog filter disabled: anime_dataset.csv is not available yet")

rows_read = 0
rows_written = 0
unique_users = set()
unique_anime = set()
first_chunk = True

if RATINGS_PROCESSED_FILE.exists():
    RATINGS_PROCESSED_FILE.unlink()
    print("Removed existing processed ratings file:", RATINGS_PROCESSED_FILE)

for chunk_idx, chunk in enumerate(pd.read_csv(raw_ratings_path, chunksize=CHUNK_SIZE), start=1):
    chunk.columns = chunk.columns.str.strip()
    chunk = chunk.rename(
        columns={
            "user_id": "userID",
            "anime_id": "animeID",
        }
    )

    required = {"userID", "animeID", "rating"}
    missing = required - set(chunk.columns)
    if missing:
        raise ValueError(f"Raw ratings chunk is missing required columns: {sorted(missing)}")

    chunk = chunk[["userID", "animeID", "rating"]].dropna()
    chunk["userID"] = chunk["userID"].astype("int32")
    chunk["animeID"] = chunk["animeID"].astype("int32")
    chunk["rating"] = chunk["rating"].astype("int8")

    rows_read += len(chunk)

    if catalog_ids is not None:
        chunk = chunk[chunk["animeID"].isin(catalog_ids)]

    rows_written += len(chunk)
    unique_users.update(chunk["userID"].unique().tolist())
    unique_anime.update(chunk["animeID"].unique().tolist())

    chunk.to_csv(
        RATINGS_PROCESSED_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )
    first_chunk = False

    print(
        f"Chunk {chunk_idx:03d} | read={rows_read:,} | "
        f"written={rows_written:,} | users={len(unique_users):,} | anime={len(unique_anime):,}"
    )

summary = {
    "updated_at": now_iso(),
    "source_dataset": KAGGLE_DATASET,
    "source_file": "rating_complete.csv",
    "source_collection_window": "2020-02-26 to 2020-03-20, per Kaggle dataset documentation",
    "processed_file": str(RATINGS_PROCESSED_FILE),
    "catalog_filter_enabled": catalog_ids is not None,
    "catalog_anime_ids": len(catalog_ids) if catalog_ids is not None else None,
    "rows_read": int(rows_read),
    "rows_written": int(rows_written),
    "unique_users_written": int(len(unique_users)),
    "unique_anime_written": int(len(unique_anime)),
    "columns": ["userID", "animeID", "rating"],
}
atomic_write_json(RATINGS_BUILD_SUMMARY_FILE, summary)

print("Ratings processed and saved:", RATINGS_PROCESSED_FILE)
print("Build summary saved:", RATINGS_BUILD_SUMMARY_FILE)
print(summary)


Catalog filter enabled: 14,991 anime ids
Chunk 001 | read=2,000,000 | written=1,864,605 | users=10,827 | anime=10,711
Chunk 002 | read=4,000,000 | written=3,733,707 | users=21,568 | anime=11,135
Chunk 003 | read=6,000,000 | written=5,600,410 | users=32,262 | anime=11,361
Chunk 004 | read=8,000,000 | written=7,465,894 | users=42,972 | anime=11,478
Chunk 005 | read=10,000,000 | written=9,332,224 | users=53,818 | anime=11,507
Chunk 006 | read=12,000,000 | written=11,196,253 | users=64,484 | anime=11,648
Chunk 007 | read=14,000,000 | written=13,059,513 | users=75,154 | anime=11,662
Chunk 008 | read=16,000,000 | written=14,928,556 | users=86,053 | anime=11,673
Chunk 009 | read=18,000,000 | written=16,800,129 | users=96,875 | anime=11,686
Chunk 010 | read=20,000,000 | written=18,668,801 | users=107,807 | anime=11,689
Chunk 011 | read=22,000,000 | written=20,533,192 | users=118,435 | anime=11,691
Chunk 012 | read=24,000,000 | written=22,399,893 | users=129,367 | anime=11,691
Chunk 013 | read=

## Remove Disposable Raw CSVs

After `ratings_processed.csv` exists, the raw Kaggle CSV inside this repository is no longer needed. Keeping only the processed file avoids duplicating multi-GB data in `data/raw/`.


In [4]:
if RATINGS_PROCESSED_FILE.exists():
    remove_disposable_raw_files()
else:
    print("ratings_processed.csv was not created; raw source file kept for inspection.")


Deleted disposable raw file: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\kaggle_rating_complete_source.csv
